<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 4: *Variable Selection*
##### Version Number: 4.0
---
### Contents  
> 1. *Water Demand*
> 2. *Water Supply*
> 3. *Water Supply Indexes*
> 4. *Fire Danger Indicators*
> 5. *Social Variables*
> 6. *Temporal and Geographic Varaibles*
> 7. *Export File*
---
### Notes
- This module visualizes key variables to assess their relationships with wildfire severity categories. Based on the `Categorical` target, we explore how different weather features interact and correlate with fire risk.
---
### Inputs
- `engineered_samples.csv` engineered and cleaned samples data with weather, fire, and grid data.
---
### Outputs 
- `X`,`y`,`details` - Split training data filtered from 2018 to 2024
- `pal_x`, `pal_y`, `pal_details` - Split training data from 2025 for case study
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime
import json

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

# Set consistent plotting style
sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

---

In [3]:
def plot_all(df,target,title):
    grid_kde(df)
    bar_group(df, target)
    correlation_map(pd.concat([df,target],axis=1),title)
    

### Loading Data

In [4]:
samples = pd.read_csv('../data/processed/engineered_samples.csv')

---

# One Hot Encoding

One hot encode appropriate categorical variables.

In [5]:
categorical_columns = ['dominant_province_description','dominant_section_description','Season']
one_hot = samples[categorical_columns]
samples = samples.drop(columns = ['dominant_province_description','dominant_section_description','Season'])

one_hot_samples = pd.get_dummies(
    one_hot,
    columns=categorical_columns,
    drop_first=False
).astype(int)

coded_samples = pd.concat([samples,one_hot_samples],axis=1)

## Split dataset temporarily for variable analysis

In [6]:
# Columns to drop for feature interaction analysis
text_columns = ['Sample_ID', 'Date', 'grid_id',
       'geometry', 'fire_count', 'total_fire_damage','acres','area_in_cali','maximum_x', 'minimum_y',
       'maximum_y', 'minimum_x','centroid_northing','centroid_easting','Target_Damage','Target_Ignition',
               'Target_Spread','Year','fires_missing_acres','fires_missing_damage']

numerical_data = coded_samples.drop(columns=text_columns)
detail_data = coded_samples[text_columns] # spatial data used for modeling and mapping
target = coded_samples['Target_Damage']

In [7]:
others = [
    "NDVI_mean_difference",
    "NDVI_mean_difference_has_value",
    'reservoir_count',
    'total_reservoir_level',
    'stations_missing_levels'
]

In [8]:
#numerical_data = numerical_data.drop(columns = 'total_reservoir_level')

## Scale numerical columns for easier side by side comparisons

In [9]:
scaler = MinMaxScaler()

# Scale main dataset
X_scaled = scaler.fit_transform(numerical_data)
X = pd.DataFrame(X_scaled, columns=numerical_data.columns, index=numerical_data.index)

## DIrect Water Demand Indicators

In [10]:
water_demand = [
    "Actual Evapotranspiration",
    "Solar Radiation",
    "Solar Radiation 7 Day Avg",
    "Daily Minimum Air Temperature",
    "Daily Maximum Air Temperature",
    "Daily Maximum Air Temperature 7 Day Avg",
    "Daily Minimum Air Temperature 7 Day Avg",
    "Vapor Pressure Deficit",
    "Vapor Pressure Deficit 7 Day Avg",
    "Wind Speed",
    "Wind Speed 7 Day Avg",
]

plot_all(X[water_demand],target,'Water Demand')

---

## Water Supply Indicators

In [11]:
water_supply = [
    "Precipitation",
    "Precipitation 7 Day Avg",
    "Maximum Relative Humidity",
    "Minimum Relative Humidity",
    "Maximum Relative Humidity 7 Day Avg",
    "Minimum Relative Humidity 7 Day Avg",
    "Specific Humidity",
    "100-hour Dead Fuel Moisture",
    "1000-hour Dead Fuel Moisture"
]

plot_all(X[water_supply],target,'Water Supply')

---

## Water Supply Indexes

In [12]:
water_supply_indexes = ["Standardized Precipitation Index 30-Day",
    "Standardized Precipitation Index 180-Day",
    "Standardized Precipitation Evapotranspiration Index 30-Day",
    "Standardized Precipitation Evapotranspiration Index 90-Day",
    "Standardized Precipitation Evapotranspiration Index 180-Day",
    "Palmer Drought Severity Index"
                       ]

plot_all(X[water_supply_indexes],target,'Water Supply Indexes')

## Fire Danger

In [13]:
fire_danger = [
    "Burning Index",
    "Energy Release Component"]

plot_all(X[fire_danger],target,'Fire Danger')

## Social Variables

In [14]:
social = ['total_housing', 'total_population',
       'housing_density', 'population_density', 'median_income']

plot_all(X[social],target,'Social')

## Temporal Variables

In [15]:
temporal = ['Santa_Ana_Score',    
            "Season_Fall",
            "Season_Spring",
            "Season_Summer",
            "Season_Winter"]

plot_all(X[temporal],target,'Temporal')

## Elevation

In [16]:
elevation = ['elevation_range', 'elevation_mean',
       'elevation_std', 'slope_max', 'slope_range', 'slope_mean', 'slope_std',
       'northness_mean', 'eastness_mean']

plot_all(X[elevation],target,'Elevation')

## WUI

In [17]:
WUI = ['influence_zone', 'interface_zone', 'intermix_zone']

plot_all(X[WUI],target,'WUI')

## Ecological

In [18]:
ecoregion = [ 'dominant_province_percent',
       'sum_province_area', 'sum_section_area',
       'dominant_section_percent']

plot_all(X[ecological],target,'Ecological')

## Land Cover

In [19]:
land_cover = ['forest_percent','developed_percent', 'other_percent', 'shrub_grass_percent',
       'wetlands_percent']

plot_all(X[land_cover],target,'Land Cover')

---

In [20]:
interactions = ['Wind Speed_x_100-hour Dead Fuel Moisture',
 'Vapor Pressure Deficit_x_Solar Radiation',
 'Precipitation_x_1000-hour Dead Fuel Moisture',
 'northness_mean_x_Daily Maximum Air Temperature',
 'road_density_x_forest_percent',
 'power_line_density_x_total_housing']

plot_all(X[interactions],target,'Interactions')

## Wind Slope Interaction

In [21]:
wind_slope = ['Wind Speed_x_slope_mean',
 'Wind Speed_x_slope_max',
 'Wind Speed_x_northness_mean',
 'Wind Speed_x_eastness_mean',
 'Wind Speed_x_elevation_mean',
 'Wind Speed_x_elevation_range',
 'Wind Speed 7 Day Avg_x_slope_mean',
 'Wind Speed 7 Day Avg_x_slope_max',
 'Wind Speed 7 Day Avg_x_northness_mean',
 'Wind Speed 7 Day Avg_x_eastness_mean',
 'Wind Speed 7 Day Avg_x_elevation_mean',
 'Wind Speed 7 Day Avg_x_elevation_range']

In [22]:
ecoregion.extend([
    # Province
    "dominant_province_description_American Semi-Desert and Desert",
    "dominant_province_description_California Coastal Chaparral Forest and Shrub",
    "dominant_province_description_California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow",
    "dominant_province_description_California Coastal Steppe-Mixed Forest-Redwood Forest",
    "dominant_province_description_California Dry Steppe",
    "dominant_province_description_Intermountain Semi-Desert",
    "dominant_province_description_Intermountain Semi-Desert and Desert",
    "dominant_province_description_Sierran Steppe-Mixed Forest-Coniferous Forest-Alpine Meadow",

    # Section
    "dominant_section_description_Central California Coast",
    "dominant_section_description_Central Valley Coast Ranges",
    "dominant_section_description_Colorado Desert",
    "dominant_section_description_Great Valley",
    "dominant_section_description_Klamath Mountains",
    "dominant_section_description_Modoc Plateau",
    "dominant_section_description_Mojave Desert",
    "dominant_section_description_Mono",
    "dominant_section_description_Northern California Coast",
    "dominant_section_description_Northern California Coast Ranges",
    "dominant_section_description_Northern California Interior Coast Ranges",
    "dominant_section_description_Northwestern Basin and Range",
    "dominant_section_description_Sierra Nevada",
    "dominant_section_description_Sierra Nevada Foothills",
    "dominant_section_description_Sonoran Desert",
    "dominant_section_description_Southeastern Great Basin",
    "dominant_section_description_Southern California Coast",
    "dominant_section_description_Southern California Mountains and Valleys",
    "dominant_section_description_Southern Cascades"
])

plot_all(X[wind_slope],target,'Wind Slope Interactions')

In [23]:
others = [
    "NDVI_mean_difference",
    "NDVI_mean_difference_has_value",
    #'reservoir_count',
    #'total_reservoir_level',
    #'stations_missing_levels'
]

plot_all(X[others],target,'Wind Slope Interactions')

In [24]:
feature_sets =  {
    "Water Demand": water_demand, 
    "Water Supply": water_supply, 
    "Water Supply Indexes": water_supply_indexes, 
    "Fire Danger": fire_danger,
    "Social": social, 
    "Temporal": temporal, 
    "Elevation": elevation, 
    "WUI" : WUI, 
    "Ecoregion": ecoregion, 
    "Land Cover": land_cover, 
    "Interactions": interactions, 
    "Wind Slope": wind_slope,
    "Others": others
}

In [25]:
variable_selection_output = pd.concat([X, detail_data], axis=1)

## 8. Export File

In [26]:
variable_selection_output.to_csv('../data/processed/variable_selection_output.csv', index=False)

with open('feature_sets.json', 'w') as f:
    json.dump(feature_sets, f, indent=4)


print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
